# Kinder HERC: School Board Meeting Research Detection Training Pipeline

### COMP 449 - Spring 2026
### Team: Kinder HERC

## Introduction

School board members are expected to use research, data, and evidence when making decisions that affect thousands of students. But how often does that actually happen in practice?

This notebook walks through our full **training pipeline** for a classifier that automatically detects when board members reference research, data, or statistics during recorded board meetings. The pipeline covers five automated stages (web scraping, audio extraction, transcription, chunking, and classification) plus one manual stage: human labeling.

**Districts covered:**
- **Houston ISD**: largest district in Texas (~180,000 students)
- **Katy ISD**: fast-growing suburban district west of Houston
- **Spring Branch ISD**: mid-sized urban district, northwest Houston

**What we are classifying:** Every 2-minute segment ("chunk") of a board meeting transcript is labeled **1** if a board member cites a study, dataset, report, or statistic to support a decision, and **0** otherwise.

---

**Pipeline overview:**

```
District websites
      |  web_scraping/
      v
Meeting videos (.wav)
      |  transcription/
      v
Timestamped transcripts (.txt)
      |  transcript_chunking/
      v
2-minute chunk CSVs
      |  [manual labeling]
      v
Labeled CSVs  (binary_hit: 0 or 1)
      |  research_labeling/research_chunk_pipeline/
      v
Trained logistic regression classifier
```

## Table of Contents

1. [Setup](#Setup)
2. [Step 1: Web Scraping](#Step-1:-Web-Scraping)
3. [Step 2: Audio Extraction](#Step-2:-Audio-Extraction)
4. [Step 3: Transcription](#Step-3:-Transcription-with-Parakeet-ASR)
5. [Step 4: Transcript Chunking](#Step-4:-Transcript-Chunking)
6. [Step 5: Manual Labeling](#Step-5:-Manual-Labeling)
7. [Step 6: Model Training (5-Fold CV, 10 Seeds)](#Step-6:-Model-Training)
8. [Step 7: Results and Visualization](#Step-7:-Results-and-Visualization)9. [Step 8: t-SNE Embedding Visualization](#Step-8:-t-SNE-Embedding-Visualization)

## Setup

The code below configures Python's module search path so that each sub-package in this repository (`web_scraping`, `transcription`, `transcript_chunking`, and the research pipeline utilities) can be imported directly by name. All paths are resolved relative to the repository root so the notebook works regardless of where it is opened.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Resolve key directories
REPO_ROOT       = Path('.').resolve()
PIPELINE_DIR    = REPO_ROOT / 'research_labeling' / 'research_chunk_pipeline'
TRANSCRIPT_DATA = REPO_ROOT / 'research_labeling' / 'Transcript Data'
OUTPUTS_DIR     = PIPELINE_DIR / 'outputs' / 'no_feature_selection'
AGG_DIR         = OUTPUTS_DIR / 'aggregate'

# Add repo root and pipeline directory to the import path
for p in [str(REPO_ROOT), str(PIPELINE_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'Repository root      : {REPO_ROOT}')
print(f'Labeled CSVs         : {TRANSCRIPT_DATA}')
print(f'Pre-computed outputs : {OUTPUTS_DIR}')

---
## Step 1: Web Scraping

Each of the three school districts posts its board meeting recordings on a video hosting platform called **Swagit**, either directly or embedded via `<iframe>` on its own site. Our `web_scraping` package handles both cases:

1. **`html_scrape`** fetches each district's board-meetings page, extracts `<a>` and `<iframe>` tags, and resolves Swagit video URLs.
2. **`pipeline`** orchestrates the discovery-and-download loop, filtering by meeting date.

Below we import the package, define the three `Source` objects, and run the HTML discovery step against Houston ISD's website to show a live sample of the Swagit links our scraper finds.

In [ ]:
from web_scraping.models import Source
from web_scraping.pipeline import PipelineConfig, process_source
from web_scraping.html_scrape import fetch_html, scrape_candidate_links
from web_scraping.ytdlp_runner import download_source_to_wav

# The three districts included in this study
SOURCES = [
    Source(
        district="Houston ISD",
        url="https://www.houstonisd.org/board-governance/board-meetings",
    ),
    Source(
        district="Katy ISD",
        url="https://www.katyisd.org/board/board/board-meeting-videos",
    ),
    Source(
        district="Spring Branch ISD",
        url="https://www.springbranchisd.com/about/board-of-trustees/meetings/board-meeting-videos",
    ),
]

print("Districts and board-meeting pages:")
for s in SOURCES:
    print(f"  {s.district:20s}  ->  {s.url}")

The cell below fetches the live Houston ISD board-meetings page and extracts any Swagit video links found in the HTML. Because many district sites embed Swagit inside an `<iframe>`, this direct pass may return zero results; the pipeline then automatically falls back to the iframe scraper.

In [ ]:
houston = SOURCES[0]
print(f"Fetching: {houston.url}\n")

try:
    html = fetch_html(houston.url)
    print(f"Downloaded {len(html):,} bytes of HTML.")

    candidates = scrape_candidate_links(
        page_url=houston.url,
        html=html,
        max_links=8,
    )

    if candidates:
        print(f"\nFound {len(candidates)} Swagit video link(s) on the page:")
        for url in candidates:
            print(f"  {url}")
    else:
        print(
            "\nNo direct Swagit links found on the page. This is expected.\n"
            "The pipeline will automatically search inside embedded Swagit "
            "<iframe> elements and follow paginated listing pages to collect "
            "all available meeting links."
        )
except Exception as e:
    print(f"Network request failed: {e}\n(Run with internet access to see live results.)")

The `PipelineConfig` below controls exactly which meetings to download. Setting a `cutoff` date means the scraper skips any meeting recorded before our study window, keeping storage and processing costs proportional to the meetings we actually care about.

In [ ]:
from datetime import date

cfg = PipelineConfig(
    out_root=REPO_ROOT / "School Board Meetings",
    cutoff=date(2024, 8, 1),   # only include meetings on/after August 2024
    fragment_workers=8,         # parallel HLS/DASH fragment downloads
    max_candidates=60,          # max Swagit links to attempt per page
)

print("PipelineConfig:")
print(f"  Output root    : {cfg.out_root}")
print(f"  Date cutoff    : {cfg.cutoff}")
print(f"  Max candidates : {cfg.max_candidates}")
print(f"  Frag workers   : {cfg.fragment_workers}")
print()

# Download all meetings for all three districts.
# Each meeting is 3-5 hours of video; downloading all sources takes several hours.
# for source in SOURCES:
#     ok, msg = process_source(source, cfg)
#     print(f"[{'OK' if ok else 'FAIL'}] {source.district}: {msg}")

print("Downloaded WAV files are saved to:")
print(f"  {cfg.out_root} / <District> / <YYYY-MM-DD>_<title>.wav")

---
## Step 2: Audio Extraction

Video downloading and audio conversion happen inside a single function call: `download_source_to_wav`. Internally it uses **yt-dlp** to stream-download the Swagit HLS/DASH video and pipes the result through **ffmpeg**, which resamples the audio to **16 kHz mono PCM WAV** (the format expected by Parakeet ASR). Exponential backoff retries handle transient network failures.

We do not execute the download here, as the raw WAV files are already stored on disk from a prior run. The cell below shows the audio format that `download_source_to_wav` produces and the pre-processing steps that `parakeet_transcribe` applies before running ASR inference.

In [ ]:
# download_source_to_wav() uses yt-dlp to fetch the HLS/DASH video stream
# and ffmpeg to re-encode the audio track to the format Parakeet expects.
#
# Example call (commented out - each meeting download takes 10 to 40 minutes):
# download_source_to_wav(
#     url              = "https://houston.swagit.com/video/12345",
#     district_dir     = Path("School Board Meetings/Houston ISD"),
#     district_name    = "Houston ISD",
#     cutoff           = date(2024, 8, 1),
#     fragment_workers = 8,
# )

print("Audio format produced by download_source_to_wav:")
print("  Codec     : PCM signed 16-bit little-endian")
print("  Channels  : 1 (mono)")
print("  Sample rate: 16,000 Hz")
print("  Container : WAV")
print()
print("Pre-processing applied inside parakeet_transcribe before ASR inference:")
print("  * Bandpass filter: 80 Hz to 7,500 Hz (removes HVAC rumble and high-frequency noise)")
print("  * Normalisation to -3 dBFS peak")
print("  * Re-encode to 16 kHz mono if not already in that format")

---
## Step 3: Transcription with Parakeet ASR

A board meeting WAV file is typically **3 to 5 hours** long, far beyond what an ASR model can process as a single chunk. Our `transcription` package handles this with an **overlap-and-merge** strategy:

1. The audio is sliced into **~35-second overlapping windows**.
2. Each window is transcribed in a batch using NVIDIA's **Parakeet-TDT-1.1B** model (via NeMo).
3. The model's word output at each boundary is deduplicated with a **suffix-prefix word-matching algorithm**, removing repeated words at overlap boundaries.
4. The merged text is written to a `.txt` file with **`[MM:SS-MM:SS]`** section headers every 30 seconds.

The cell below shows the public API of the transcription module. The actual `run_transcription` call is written out and commented because it requires an NVIDIA GPU and approximately 60 to 90 minutes per meeting.

In [ ]:
from transcription.parakeet_transcribe import fmt_ts, run_transcription

# fmt_ts() formats raw seconds as MM:SS or HH:MM:SS.
# run_transcription() calls this internally to label each 30-second block in the output .txt file.
demo_times = [0, 90, 3661, 7320]
print("fmt_ts() - converts seconds to a readable timestamp:")
for t in demo_times:
    print(f"  fmt_ts({t:>5}) -> '{fmt_ts(t)}'")

print()

# Transcribe a single WAV file.
# Requires NVIDIA GPU and the Parakeet model (~4 GB download on first run).
# Each 3-5 hour meeting takes approximately 60 to 90 minutes to process.
# run_transcription(
#     audio_path=Path("School Board Meetings/Houston ISD/2025-12-11_Board_Meeting.wav"),
#     output_dir=Path("transcripts/Houston ISD/"),
# )

print("Output is saved to: transcripts/<District>/<meeting_name>.txt")
print("Each output file contains 30-second blocks with [MM:SS-MM:SS] timestamp headers.")

The cell below loads one of our labeled CSVs and displays what a section of the raw Parakeet transcript file looks like. The text includes embedded `[MM:SS-MM:SS]` section headers and is fed directly into the chunking step.

In [ ]:
# Load one labeled CSV and display the raw transcript text for the first two chunks.
sample_csv = sorted(TRANSCRIPT_DATA.glob("*.csv"))[0]
sample_df  = pd.read_csv(sample_csv)

print(f"Source file : {sample_csv.name}")
print(f"Meeting     : {sample_csv.stem.split(' - ')[-1]}")
print()
print("=" * 72)
print("Sample transcript output (first two 2-minute chunks):")
print("=" * 72)
for i in range(min(2, len(sample_df))):
    row = sample_df.iloc[i]
    print(f"\n[Chunk {row['chunk_id']}  |  {row['window_start']} to {row['window_end']}]")
    print("-" * 60)
    print(row["text"][:700])
    print("...")

---
## Step 4: Transcript Chunking

The raw `.txt` transcript is a single large file with 30-second section headers. Classifying it at 30-second granularity would produce very short, often context-free text segments. Instead, the `transcript_chunking` package groups consecutive 30-second sections into **fixed 2-minute windows** and writes the result to a CSV.

Each row in the output CSV represents one chunk:

| chunk_id | window_start | window_end | text |
|----------|-------------|------------|------|
| 0 | 00:00:00 | 00:02:00 | [00:00-00:30]\nText... |
| 1 | 00:02:00 | 00:04:00 | [02:00-02:30]\nText... |

Below we demonstrate `parse_segments` and `chunk_by_time` on a short synthetic transcript, then show how the same functions process a real labeled meeting.

In [ ]:
from transcript_chunking.create_chunks import (
    parse_segments,
    chunk_by_time,
    chunk_transcript,
    time_to_seconds,
    seconds_to_hhmmss,
)

print("time_to_seconds / seconds_to_hhmmss examples:")
print(f"  time_to_seconds('01:30')    -> {time_to_seconds('01:30')} seconds")
print(f"  time_to_seconds('01:02:30') -> {time_to_seconds('01:02:30')} seconds")
print(f"  seconds_to_hhmmss(120)      -> '{seconds_to_hhmmss(120)}'")
print(f"  seconds_to_hhmmss(3661)     -> '{seconds_to_hhmmss(3661)}'")

We pass a short synthetic transcript (8 x 30-second blocks) through `parse_segments` to show how the function detects timestamp headers and extracts the text for each 30-second window.

In [ ]:
SAMPLE_TRANSCRIPT = """[00:00-00:30]
Good evening. This meeting is now convened at 5:02 PM. A quorum of the board
is present in the board auditorium.

[00:30-01:00]
I would like to welcome our elected officials in the audience tonight.
We have Trustee Rodriguez, Trustee Gomez, and Trustee-elect Mora.

[01:00-01:30]
Please rise for the pledge of allegiance led by our JROTC cadet tonight.

[01:30-02:00]
I pledge allegiance to the flag of the United States of America.

[02:00-02:30]
The first item on the agenda is a presentation on student academic performance.

[02:30-03:00]
According to our district research division, math proficiency rose 7 percentage
points year-over-year, from 61% to 68%, as measured by STAAR assessments.

[03:00-03:30]
This improvement is supported by an independent evaluation study conducted in
fall 2024 by the University of Houston's Kinder Institute.

[03:30-04:00]
The study used a difference-in-differences design comparing HISD students to a
matched cohort of students in comparable districts across the state.
"""

segments = parse_segments(SAMPLE_TRANSCRIPT)
print(f"parse_segments found {len(segments)} 30-second segment(s):\n")
for seg in segments:
    preview = seg["raw"].splitlines()[1][:60] if len(seg["raw"].splitlines()) > 1 else ""
    print(f"  [{seg['start']:>4}s to {seg['end']:>4}s]  {preview}...")

Now `chunk_by_time` groups those 30-second segments into **2-minute windows**, concatenating the raw text blocks within each window.

In [ ]:
chunks = chunk_by_time(segments, window_seconds=120)  # 2 minutes

chunks_df = pd.DataFrame(chunks)[["chunk_id", "window_start", "window_end", "text"]]
# Compact text preview: strip timestamp headers and collapse whitespace
chunks_df["preview"] = (
    chunks_df["text"]
    .str.replace(r"\[\d{2}:\d{2}-\d{2}:\d{2}\]\n", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str[:80]
)

print(f"chunk_by_time produced {len(chunks_df)} chunk(s) (2-minute windows):\n")
print(chunks_df[["chunk_id", "window_start", "window_end", "preview"]].to_string(index=False))

And here is a real labeled chunk CSV, the actual output format that the labeling step and classifier consume.

In [ ]:
# Load a real labeled CSV and display its structure
real_csv = sorted(TRANSCRIPT_DATA.glob("*.csv"))[3]   # fourth file alphabetically
real_df  = pd.read_csv(real_csv)[["chunk_id", "window_start", "window_end", "binary_hit"]]

print(f"File: {real_csv.name}\n")
print(real_df.head(10).to_string(index=False))
print(f"\n  Total chunks : {len(real_df)}")
print(f"  Positives    : {real_df['binary_hit'].sum()}  ({real_df['binary_hit'].mean()*100:.1f}%)")
print(f"  Negatives    : {(real_df['binary_hit']==0).sum()}  ({(1-real_df['binary_hit'].mean())*100:.1f}%)")

---
## Step 5: Manual Labeling

After chunking, each 2-minute segment was **manually reviewed and labeled** by a team member. The labeling criterion was strict and binary:

| Label | Meaning |
|-------|----------|
| **1** | A board member, superintendent, or presenter explicitly references a piece of research, a data source, a report, or a statistic in the context of a district decision or policy discussion. |
| **0** | The chunk contains no such reference. This includes procedural items, public comment, oaths, pledges, and general discussion without cited evidence. |

We covered **49 board meetings** across the three districts, with meetings dating from August 2024 through January 2026, yielding a dataset with substantial class imbalance (most of any meeting's runtime is procedural, not research-driven). This imbalance directly informed our choice of **F2 score** as the threshold-selection criterion, which weights recall twice as heavily as precision to reduce false negatives.

The cells below load the complete labeled dataset and summarize its characteristics.

In [ ]:
from data_utils import find_transcript_csvs, load_all_transcripts

csv_paths   = find_transcript_csvs(TRANSCRIPT_DATA)
combined_df = load_all_transcripts(TRANSCRIPT_DATA)

n_transcripts = combined_df["transcript_id"].nunique()
n_chunks      = len(combined_df)
n_positives   = int(combined_df["binary_hit"].sum())
n_negatives   = n_chunks - n_positives
pos_rate      = n_positives / n_chunks

print("=" * 50)
print("Labeled dataset summary")
print("=" * 50)
print(f"  Meetings (transcripts) : {n_transcripts}")
print(f"  Total 2-min chunks     : {n_chunks:,}")
print(f"  Positive chunks  (1)   : {n_positives:,}  ({pos_rate*100:.1f}%)")
print(f"  Negative chunks  (0)   : {n_negatives:,}  ({(1-pos_rate)*100:.1f}%)")
print(f"  Imbalance ratio        : 1 positive per {n_negatives/n_positives:.1f} negatives")

In [ ]:
%matplotlib inline

BLUE  = "#2C7BB6"
RED   = "#E22808"
GREEN = "#1A9641"
LIGHT = "#F7F7F7"
WHITE = "#FFFFFF"

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.patch.set_facecolor(WHITE)
fig.suptitle("Labeled Dataset Overview", fontsize=13, fontweight="bold", y=1.01)

# Left: overall class distribution
ax = axes[0]
ax.set_facecolor(LIGHT)
ax.grid(axis="y", color=WHITE, linewidth=1.5, zorder=0)
counts = combined_df["binary_hit"].value_counts().sort_index()
bars   = ax.bar(
    ["Negative (0)", "Positive (1)"],
    counts.values,
    color=[BLUE, RED],
    edgecolor=WHITE,
    width=0.5,
    zorder=3,
)
for bar, val in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 8,
        f"{val:,}",
        ha="center", fontsize=11, fontweight="bold",
    )
ax.set_title("Class Distribution (all meetings)", fontsize=10)
ax.set_ylabel("Chunk count", fontsize=10)
for spine in ax.spines.values():
    spine.set_visible(False)

# Right: per-meeting positive chunk count
ax2 = axes[1]
ax2.set_facecolor(LIGHT)
ax2.grid(axis="y", color=WHITE, linewidth=1.5, zorder=0)
per_meeting = combined_df.groupby("transcript_id")["binary_hit"].sum().sort_values()
ax2.hist(per_meeting.values, bins=12, color=GREEN, edgecolor=WHITE, zorder=3)
ax2.axvline(
    per_meeting.mean(), color=RED, linewidth=2, linestyle="--",
    label=f"Mean = {per_meeting.mean():.1f}",
)
ax2.set_title("Positive Chunks per Meeting", fontsize=10)
ax2.set_xlabel("# of positive chunks in the meeting", fontsize=10)
ax2.set_ylabel("# of meetings", fontsize=10)
ax2.legend(fontsize=9)
for spine in ax2.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

Below are concrete examples of positive and negative chunks, illustrating what our annotators were looking for. Positive chunks contain explicit references to studies, reports, or data used in the board's deliberations; negative chunks cover procedural matters, public comment, and other non-evidential content.

In [ ]:
import re

pos_examples = combined_df[combined_df["binary_hit"] == 1].sample(2, random_state=7)
neg_examples = combined_df[combined_df["binary_hit"] == 0].sample(2, random_state=7)

print("-" * 72)
print("POSITIVE chunks  (binary_hit = 1)  - research, data, or statistics cited")
print("-" * 72)
for _, row in pos_examples.iterrows():
    print(f"\n[{row['window_start']} to {row['window_end']}]")
    cleaned = row["text"].replace("\r\n", " ").replace("\n", " ")
    cleaned = re.sub(r"\[\d{1,2}:\d{2}[\u2013-]\d{1,2}:\d{2}\]", "", cleaned).strip()
    print(cleaned[:450] + "...")

print()
print("-" * 72)
print("NEGATIVE chunks  (binary_hit = 0)  - procedural or non-evidential content")
print("-" * 72)
for _, row in neg_examples.iterrows():
    print(f"\n[{row['window_start']} to {row['window_end']}]")
    cleaned = row["text"].replace("\r\n", " ").replace("\n", " ")
    cleaned = re.sub(r"\[\d{1,2}:\d{2}[\u2013-]\d{1,2}:\d{2}\]", "", cleaned).strip()
    print(cleaned[:450] + "...")

---
## Step 6: Model Training

With a fully labeled dataset in hand, we train a **logistic regression** classifier.
Rather than randomly splitting chunks, we split at the **transcript level**: all chunks
from a given meeting always stay together in the same fold. This prevents data leakage
caused by adjacent chunks sharing speakers, topics, and phrasing.

### Architecture choices

| Component | Choice | Reason |
|-----------|--------|--------|
| Embeddings | `all-mpnet-base-v2` (768-d) | Strong semantic representations; runs on CPU |
| Classifier | Logistic regression | Interpretable; calibrated probabilities |
| Tuned params | `C` (regularisation), decision threshold | Only two parameters; avoids overfitting |
| Threshold criterion | Grid search (0.1 to 0.9), maximising **F2** | Recall weighted twice as heavily as precision |

We experimented with LASSO-based feature selection (AIC and BIC criteria) but found that
training directly on the full 768-d MPNet embeddings without feature selection produced
the best held-out F2. The tuned hyperparameters are therefore just **C** and the
**decision threshold**.

### Experimental setup
- **10 independent seeds** (1 to 10) to measure variance
- **5-fold grouped CV** per seed -> 50 total held-out evaluations
- Within each fold: an inner 80/20 train/val split selects the best `C` and decision threshold

The call below is what was executed to produce all pre-computed results shown in Step 7.
Re-running it takes approximately 25 to 40 minutes because embedding 49 meetings requires
several passes through the sentence-transformer model.

In [ ]:
import subprocess

# Run the multi-seed experiment.
# This embeds all chunks with all-mpnet-base-v2, runs 5-fold CV for each of
# the 10 seeds, and aggregates metrics across all 50 fold evaluations.
# Takes approximately 25 to 40 minutes on a modern CPU.
# subprocess.run(
#     [
#         sys.executable, str(PIPELINE_DIR / 'run_experiments.py'),
#         '--experiment-name',     'no_feature_selection',
#         '--transcript-data-dir', str(TRANSCRIPT_DATA),
#         '--seeds', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10',
#         '--n-folds',             '5',
#         '--base-output-dir',     str(PIPELINE_DIR / 'outputs'),
#     ],
#     cwd=str(PIPELINE_DIR),
#     check=True,
# )

print('Results are organized under:')
print(f'  {OUTPUTS_DIR}')
print()
print('  seed_<n>/')
print('    cv_results.json              - per-fold metrics for this seed')
print('    fold_<i>_test_predictions.csv  - predicted labels and probabilities')
print('    fold_<i>_false_positives.csv   - predicted 1, true 0')
print('    fold_<i>_false_negatives.csv   - predicted 0, true 1')
print()
print('  aggregate/')
print('    all_fold_results.csv           - one row per (seed, fold)')
print('    aggregate_summary.json         - mean, std, min, max per metric')
print('    metrics_barchart.png           - bar chart of mean and std for 5 metrics')
print('    confusion_matrix.png           - confusion matrix summed across all folds')
print('    threshold_frequency.png        - how often each threshold was selected')
print('    c_frequency.png                - how often each C value was selected')

In [ ]:
# Load and display the per-fold results from seed 1 to show what one CV run looks like.
seed1_results_path = OUTPUTS_DIR / "seed_1" / "cv_results.json"
with open(seed1_results_path, encoding="utf-8") as f:
    seed1 = json.load(f)

print(f"Seed 1 - {seed1['config']['n_folds']}-fold grouped CV")
print(f"  Transcripts  : {seed1['config']['n_transcripts']}")
print(f"  Chunks       : {seed1['config']['n_chunks']:,}")
print(f"  Positives    : {seed1['config']['n_positives']} ({seed1['config']['n_positives']/seed1['config']['n_chunks']*100:.1f}%)")
print(f"  Feature mode : {seed1['config']['feature_mode']}")
print()

fold_rows = []
for fr in seed1["fold_results"]:
    fold_rows.append({
        "Fold"            : fr["fold"],
        "Test transcripts": fr["n_test_transcripts"],
        "Best C"          : fr["best_c"],
        "Threshold"       : round(fr["best_threshold"], 2),
        "Recall"          : round(fr["test_metrics"]["recall"], 3),
        "Precision"       : round(fr["test_metrics"]["precision"], 3),
        "F1"              : round(fr["test_metrics"]["f1"], 3),
        "F2"              : round(fr["test_metrics"]["f2"], 3),
        "Avg Prec"        : round(fr["test_metrics"]["average_precision"], 3),
    })

fold_df = pd.DataFrame(fold_rows)
print(fold_df.to_string(index=False))

print()
agg = seed1["aggregate"]
print("Seed 1 aggregate:")
for k in ["recall", "precision", "f1", "f2", "average_precision"]:
    print(f"  {k:20s}  {agg[k+'_mean']:.3f} +/- {agg[k+'_std']:.3f}")

---
## Step 7: Results and Visualization

The cells below aggregate results across all 10 seeds x 5 folds = **50 held-out evaluations**. We report each metric as mean +/- standard deviation to capture both typical performance and the variance introduced by different random splits.

In [ ]:
with open(AGG_DIR / 'aggregate_summary.json', encoding='utf-8') as f:
    summary = json.load(f)

print(f"Experiment        : {summary['experiment']}")
print(f"Seeds             : {summary['seeds']}")
print(f"Folds per seed    : {summary['n_folds']}")
print(f"Total folds       : {summary['n_total_folds']}")
print(f"Feature mode      : {summary['feature_mode']}")
print(f"Feature selection : {summary['use_feature_selection']}")

In [ ]:
# Build a clean metrics summary table
metrics_rows = []
for key, label in [
    ("recall",            "Recall"),
    ("precision",         "Precision"),
    ("f1",                "F1"),
    ("f2",                "F2  (optimisation target)"),
    ("average_precision", "Average Precision (AUC-PR)"),
]:
    m = summary["metrics"][key]
    metrics_rows.append({
        "Metric" : label,
        "Mean"   : f"{m['mean']:.3f}",
        "Std"    : f"{m['std']:.3f}",
        "Min"    : f"{m['min']:.3f}",
        "Max"    : f"{m['max']:.3f}",
    })

metrics_df = pd.DataFrame(metrics_rows)
print(f"Aggregate test metrics across {summary['n_total_folds']} fold evaluations")
print(f"({len(summary['seeds'])} seeds x {summary['n_folds']} folds)\n")
print(metrics_df.to_string(index=False))
print()
print("Threshold selection summary:")
print(f"  Mean threshold   : {summary['threshold_mean']:.3f}")
print(f"  Median threshold : {summary['threshold_median']:.3f}")
print(f"  Frequencies      : {summary['threshold_selection_counts']}")

The bar chart below, generated by `run_experiments._plot_metrics`, shows the mean
test score for each metric with +/- 1 standard deviation error bars. The high
recall (~0.886) reflects our F2-based threshold selection, which deliberately
accepts more false positives to avoid missing real research citations.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
img = mpimg.imread(AGG_DIR / "metrics_barchart.png")
ax.imshow(img)
ax.axis("off")
plt.tight_layout(pad=0)
plt.show()

The confusion matrix below accumulates raw TP / FP / FN / TN counts summed across **all 50 fold evaluations**. This gives a population-level view of the model's error modes: how many research mentions were correctly flagged, how many were missed (false negatives), and how many non-research chunks were incorrectly flagged (false positives).

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
img = mpimg.imread(AGG_DIR / "confusion_matrix.png")
ax.imshow(img)
ax.axis("off")
plt.tight_layout(pad=0)
plt.show()

The two frequency charts below show how consistently the inner-validation step selected the same decision threshold and regularisation strength `C` across all 50 folds. High concentration around a few values indicates stable, reproducible model selection. The pipeline is not sensitive to a specific random split.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

thresh_img = mpimg.imread(AGG_DIR / "threshold_frequency.png")
axes[0].imshow(thresh_img)
axes[0].axis("off")
axes[0].set_title("Decision Threshold Selection", fontsize=11)

c_img = mpimg.imread(AGG_DIR / "c_frequency.png")
axes[1].imshow(c_img)
axes[1].axis("off")
axes[1].set_title("Regularisation C Selection", fontsize=11)

plt.tight_layout(pad=0.5)
plt.show()

Finally, we load the per-fold results CSV (one row per seed/fold pair) to show the full distribution of test scores across all 50 evaluations and confirm no single fold dominates the aggregate.

In [ ]:
all_folds_df = pd.read_csv(AGG_DIR / "all_fold_results.csv")

print(f"all_fold_results.csv shape: {all_folds_df.shape}")
print()
print(all_folds_df[
    ["seed", "fold", "best_c", "best_threshold",
     "test_recall", "test_precision", "test_f1", "test_f2", "test_average_precision"]
].head(15).to_string(index=False))

print()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
fig.patch.set_facecolor(WHITE)

PURPLE = "#9B59B6"

# Recall distribution across all 50 folds
axes[0].set_facecolor(LIGHT)
axes[0].hist(all_folds_df["test_recall"], bins=15, color=RED, edgecolor=WHITE, zorder=3)
axes[0].axvline(all_folds_df["test_recall"].mean(), color="black", lw=2, ls="--",
                label=f"Mean = {all_folds_df['test_recall'].mean():.3f}")
axes[0].set_title("Test Recall - 50 folds", fontsize=10)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=9)
for spine in axes[0].spines.values():
    spine.set_visible(False)

# F2 distribution across all 50 folds
axes[1].set_facecolor(LIGHT)
axes[1].hist(all_folds_df["test_f2"], bins=15, color=PURPLE, edgecolor=WHITE, zorder=3)
axes[1].axvline(all_folds_df["test_f2"].mean(), color="black", lw=2, ls="--",
                label=f"Mean = {all_folds_df['test_f2'].mean():.3f}")
axes[1].set_title("Test F2 - 50 folds", fontsize=10)
axes[1].set_xlabel("F2 score")
axes[1].set_ylabel("Count")
axes[1].legend(fontsize=9)
for spine in axes[1].spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

---
## Step 8: t-SNE Embedding Visualization

To understand *where* in embedding space research citations live, we visualize all
49 meetings using **t-SNE** applied on top of the same `all-mpnet-base-v2` 768-d
sentence embeddings used by the classifier. The reduction pipeline is:

1. Embed every labeled chunk with `all-mpnet-base-v2` (768 dimensions)
2. Reduce to 50 dimensions with **PCA** (speeds up t-SNE, removes noise)
3. Reduce to 2 dimensions with **t-SNE** (`perplexity=40`, `max_iter=1000`, `init="pca"`)

Positive clusters (research mentions) are then identified with **HDBSCAN** and
labeled A through Q by topic. The five plots below each highlight a different
facet of the same 2-D projection.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

PLOTS_DIR = PIPELINE_DIR / 'plots'

# Generate all five t-SNE plots from scratch.
# Requires embedding 49 meetings (~15 min on CPU) then running t-SNE reduction.
# subprocess.run(
#     [
#         sys.executable,
#         str(PIPELINE_DIR / 'results_visualization_scripts' / 'plot_embeddings_extended.py'),
#         '--transcript-data-dir', str(TRANSCRIPT_DATA),
#     ],
#     cwd=str(PIPELINE_DIR),
#     check=True,
# )

print(f'Pre-generated plots directory: {PLOTS_DIR}')
print('Files:')
for p in sorted(PLOTS_DIR.glob('*.png')):
    print(f'  {p.name}  ({p.stat().st_size // 1024} KB)')

### Plot 1: Full Dataset (Binary Labels)

Each point is one 2-minute chunk. **Red** = research/data citation (positive),
**blue** = no citation (negative). The projection reveals that positive chunks
do not form a single tight cluster; they are distributed across several
semantically distinct regions of the embedding space, which motivates examining
each cluster individually.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
img = mpimg.imread(PLOTS_DIR / 'full_dataset_tsne.png')
ax.imshow(img)
ax.axis('off')
ax.set_title('t-SNE: Full Dataset - Binary Labels', fontsize=12, fontweight='bold')
plt.tight_layout(pad=0)
plt.show()

### Plot 2: Colored by District

The same 2-D projection, now colored by district (Houston ISD, Katy ISD,
Spring Branch ISD). **Stars** mark positive chunks. This view shows whether
districts occupy distinct regions of semantic space or whether their content
overlaps substantially, which would indicate that the classifier can generalize
across districts rather than memorizing district-specific phrasing.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
img = mpimg.imread(PLOTS_DIR / 'tsne_by_district.png')
ax.imshow(img)
ax.axis('off')
ax.set_title('t-SNE: Colored by District (stars = positive)', fontsize=12, fontweight='bold')
plt.tight_layout(pad=0)
plt.show()

### Plot 3: Positive Neighborhoods (HDBSCAN Clusters)

We ran **HDBSCAN** on the positive-only subset to identify coherent topic clusters.
Each cluster is given a letter label (A through Q) and a human-readable topic name
based on the dominant vocabulary in that cluster. The table below lists all
identified neighborhoods.

| Label | Topic |
|-------|-------|
| A | Academic achievement data |
| B | Budget and financial reports |
| C | Curriculum adoption studies |
| D | Demographic and enrollment statistics |
| E | External evaluation and audit findings |
| F | Federal and state compliance data |
| G | Grant and funding research |
| H | Health and safety statistics |
| I | Infrastructure and facilities data |
| J | Job market and workforce data |
| K | K-12 program effectiveness research |
| L | Legal and policy citations |
| M | Mental health and student wellness research |
| N | National benchmark comparisons |
| O | Outcome-based accountability measures |
| P | Parent and community survey data |
| Q | Qualitative case study references |

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
img = mpimg.imread(PLOTS_DIR / 'tsne_positive_neighborhoods.png')
ax.imshow(img)
ax.axis('off')
ax.set_title('t-SNE: Positive Neighborhoods (HDBSCAN, A-Q labels)', fontsize=12, fontweight='bold')
plt.tight_layout(pad=0)
plt.show()

### Plot 4: Neighborhoods with All Negatives

The same neighborhood ellipses overlaid on the **full dataset** (positives +
negatives). Negative chunks that fall inside a positive neighborhood ellipse are
the hardest cases for the classifier: they use similar vocabulary to research
citations but lack an explicit citation. This view highlights the overlap between
classes in semantic space.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
img = mpimg.imread(PLOTS_DIR / 'tsne_neighborhoods_with_negatives.png')
ax.imshow(img)
ax.axis('off')
ax.set_title('t-SNE: Neighborhoods with Negatives Overlay', fontsize=12, fontweight='bold')
plt.tight_layout(pad=0)
plt.show()

### Plot 5: Neighborhoods Colored by Transcript

Finally, each positive chunk is colored by which transcript (meeting) it came
from. If positive chunks from the same meeting cluster together, it would suggest
that individual meetings have a strong stylistic effect on the embeddings.
Dispersion across meetings within each cluster confirms that the semantic
neighborhoods reflect genuine topic structure, not meeting-specific artifacts.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
img = mpimg.imread(PLOTS_DIR / 'tsne_neighborhoods_by_transcript.png')
ax.imshow(img)
ax.axis('off')
ax.set_title('t-SNE: Neighborhoods Colored by Transcript', fontsize=12, fontweight='bold')
plt.tight_layout(pad=0)
plt.show()

### Neighborhood CSV

The `positive_neighborhoods.csv` file stores the cluster assignment, 2-D t-SNE
coordinates, and raw text for every positive chunk. The cell below loads it and
prints the chunk count per neighborhood, giving a quantitative view of cluster sizes.

In [ ]:
neighborhoods_csv = PLOTS_DIR / 'positive_neighborhoods.csv'
nbhd_df = pd.read_csv(neighborhoods_csv)

print(f'Loaded {len(nbhd_df)} positive chunks with neighborhood assignments.')
print(f'Columns: {list(nbhd_df.columns)}')
print()

counts = nbhd_df.groupby(['neighborhood_label', 'neighborhood']).size().reset_index(name='chunks')
counts = counts.sort_values('neighborhood_label').reset_index(drop=True)
print('Chunk counts per neighborhood:')
print(counts.to_string(index=False))
print()
print(f"Transcripts represented: {nbhd_df['transcript_id'].nunique()}")
print(f"Unlabeled (noise, HDBSCAN label -1): {(nbhd_df['neighborhood_label'] == -1).sum()}")

---
## Conclusion

This notebook walked through the complete training pipeline for the Kinder HERC research-detection classifier:

| Stage | Tool / Module | Output |
|-------|--------------|--------|
| Web scraping | `web_scraping` | Swagit meeting URLs |
| Audio extraction | `web_scraping.ytdlp_runner` | `.wav` files (16 kHz mono) |
| Transcription | `transcription.parakeet_transcribe` | Timestamped `.txt` transcripts |
| Chunking | `transcript_chunking.create_chunks` | 2-minute chunk CSVs |
| Labeling | Manual annotation | `binary_hit` column |
| Training | `research_chunk_pipeline.run_experiments` | Logistic regression + threshold |

**Key results** (10 seeds x 5-fold CV, no feature selection, tuned params: C and threshold):

- **Recall: 0.886 +/- 0.071**: the model catches ~88.6% of all research/data citations in held-out meetings.
- **Precision: 0.449 +/- 0.060**: roughly 1 in 2 flagged chunks is a true positive, acceptable given our recall-first objective.
- **F2: 0.735 +/- 0.043**: strong and stable across all 50 evaluations.
- **Avg. Precision: 0.755 +/- 0.048**: the ranking quality of the classifier is consistently above 0.75.

The trained model is used by the `app` package to score new board meetings and produce highlighted Word documents that flag the time windows most likely to contain research or data citations.